## 依赖导入

本例集合演示非线性规划（二次/凹二次规划、均值-方差模型、L1 最小化、双目标规划与选址-运输问题）的两种求解途径：用 `cvxpy` 建模凸优化问题（`CVXOPT`/`GLPK_MI` 求解器），用 `scipy.optimize.minimize` 求解带约束的一般非线性规划。统一导入 `cvxpy`、`numpy`、`pandas` 与 `minimize`。

In [ ]:
import cvxpy as cp
import numpy as np
import pandas as pd
from scipy.optimize import minimize

## 3.1  带约束二次规划（CVXPY）

用 CVXPY 求解 $\min \sum_j x_j^2 - 4x_1 + 4 = \min\,(x_1-2)^2 + x_2^2$，约束 $-x_1 + x_2 \le 2$、$x_1^2 - x_2 + 1 \le 0$，且 $x \ge 0$（`pos=True`）。选用 `CVXOPT` 求解器，输出最优值与四舍五入的最优解（源程序 ex5_1.py）。

In [ ]:
x = cp.Variable(2, pos=True)
obj = cp.Minimize(sum(x**2) - 4 * x[0] + 4)
con = [-x[0] + x[1] - 2 <= 0, x[0] ** 2 - x[1] + 1 <= 0]
prob = cp.Problem(obj, con)
prob.solve(solver="CVXOPT")
print("最优值为:", round(prob.value, 4))
print("最优解为：\n", np.round(x.value, 4))

## 3.3  凹二次最大化（CVXPY）

求解 $\max\ x^T C_2 x + c_1^T x$（$C_2$ 负定，为凹二次规划），约束 $Ax \le b$、$x \ge 0$；用 `cp.quad_form` 表达二次项、`CVXOPT` 求解，输出最优解与最优值（源程序 ex5_3.py）。

In [ ]:
c2 = np.array([[-1, -0.15], [-0.15, -2]])
c1 = np.array([98, 277])
a = np.array([[1, 1], [1, -2]])
b = np.array([100, 0])
x = cp.Variable(2, pos=True)
obj = cp.Maximize(cp.quad_form(x, c2) + c1 @ x)
con = [a @ x <= b]
prob = cp.Problem(obj, con)
prob.solve(solver="CVXOPT")
print("最优解为：", np.round(x.value, 4))
print("最优值为：", round(prob.value, 4))

## 3.4  均值-方差投资组合（CVXPY）

读取 `data/5_4.txt` 三种资产年收益数据，逐列求平均得年平均收益 `mu`，对转置数据求协方差得矩阵 `F`。建立两个二次规划：① 最小化方差 `quad_form(x, F)`，约束期望收益 $\mu^T x \ge 0.15$、权重和 $\sum x = 1$；② 最大化期望收益，约束权重和为 1、方差不超过 0.09。分别用 `CVXOPT` 求解并输出最优值与最优解（源程序 ex5_4.py）。

In [ ]:
a = np.loadtxt("data/5_4.txt")
mu = a.mean(axis=0)
F = np.cov(a.T)
x = cp.Variable(3, pos=True)
ob1 = cp.Minimize(cp.quad_form(x, F))
con1 = [mu @ x >= 0.15, sum(x) == 1]
prob1 = cp.Problem(ob1, con1)
prob1.solve(solver="CVXOPT")
print("最优值为：", round(prob1.value, 4))
print("最优解为：", np.round(x.value, 4))
ob2 = cp.Maximize(mu @ x)
con2 = [sum(x) == 1, cp.quad_form(x, F) <= 0.09]
prob2 = cp.Problem(ob2, con2)
prob2.solve(solver="CVXOPT")
print("最优值为：", round(prob2.value, 4))
print("最优解为：", np.round(x.value, 4))

## 3.5  用 scipy minimize 求解二次规划

对与 3.3 相同的凹二次目标 $f(x)=x^T C_2 x + c_1^T x$，改用 `scipy.optimize.minimize` 求最小值：以 $b - Ax \ge 0$ 表示不等式约束 `con`，用 `bd` 给定 $x \ge 0$，随机初值 `np.random.randn(2)` 出发，打印完整结果 `res`（源程序 ex5_5.py）。

In [ ]:
c2 = np.array([[-1, -0.15], [-0.15, -2]])
c1 = np.array([98, 277])
a = np.array([[1, 1], [1, -2]])
b = np.array([100, 0])


def obj(x):
    return x @ c2 @ x + c1 @ x


con = {"type": "ineq", "fun": lambda x: b - a @ x}
bd = [(0, np.inf) for i in range(a.shape[1])]
res = minimize(obj, np.random.randn(2), constraints=con, bounds=bd)  # noqa: NPY002
print(res)

## 3.6  含等式与不等式约束的非线性规划

目标 $\min \sum x_i^2 + 8$。`constr1` 给出两个不等式约束 $x_1^2 - x_2 + x_3^2 \ge 0$、$20 - x_1 - x_2^2 - x_3^2 \ge 0$；`constr2` 给出两个等式约束 $-x_1 - x_2^2 + 2 = 0$、$x_2 + 2x_3^2 - 3 = 0$。用 `minimize`（SLSQP）求解并打印 `res`（源程序 ex5_6.py）。

In [ ]:
def obj(x):
    return sum(x**2) + 8
def constr1(x):
    x1, x2, x3 = x
    return [x1**2 - x2 + x3**2, 20 - x1 - x2**2 - x3**2]
def constr2(x):
    x1, x2, x3 = x
    return [-x1 - x2**2 + 2, x2 + 2 * x3**2 - 3]
con1 = {"type": "ineq", "fun": constr1}
con2 = {"type": "eq", "fun": constr2}
con = [con1, con2]
bd = [(0, np.inf) for i in range(3)]
res = minimize(obj, np.random.randn(3), constraints=con, bounds=bd)  # noqa: NPY002
print(res)

## 3.7  L1 范数最小化的两种写法

求解等式约束 $Ax = b$ 下的 $\min \sum_{j} c_j |x_j|$（$c = 1..4$）。本单元用 `cp.abs(x)` 直接对绝对值建模，选用 `GLPK_MI` 混合整数求解器，输出最优值与最优解（源程序 ex5_7_1.py）。

In [ ]:
c = np.arange(1, 5)
a = np.array([[1, -1, -1, 1], [1, -1, 1, -3], [1, -1, -2, 3]])
b = np.array([0, 1, -1 / 2])
x = cp.Variable(4)
obj = cp.Minimize(c @ cp.abs(x))
con = [a @ x == b]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)

改写为线性规划：引入 $u, v \ge 0$ 令 $x = u - v$，则 $|x| = u + v$，目标变为 $\min\ c^T(u+v)$、约束 $A(u-v) = b$。用 `GLPK_MI` 求解后分别打印 $u, v$，并还原原问题最优解 $u - v$（源程序 ex5_7_2.py）。

In [ ]:
c = np.arange(1, 5)
a = np.array([[1, -1, -1, 1], [1, -1, 1, -3], [1, -1, -2, 3]])
b = np.array([0, 1, -1 / 2])
u = cp.Variable(4, pos=True)
v = cp.Variable(4, pos=True)
obj = cp.Minimize(c @ (u + v))
con = [a @ (u - v) == b]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", u.value, "\n", v.value)
print("原问题的最优解为：", u.value - v.value)

## 3.8  两料场选址-运输非线性规划

读取 `data/5_8.txt`（依次为工地坐标、需求等），构造决策向量 $[x_1,x_2,y_1,y_2,z_{ij}]$：目标 `obj` 为各工地到两料场的加权运输距离之和；等式约束保证每个工地运量满足需求 $c$，不等式约束保证每个料场运量不超过 $e=20$；`bd` 限定各分量 $\ge 0$，随机初值出发用 `minimize` 求解，打印 `res` 并输出两个料场坐标与运输量表（源程序 ex5_8.py）。

In [ ]:
d = np.loadtxt("data/5_8.txt")
a = d[0]
b = d[1]
c = d[2]
e = np.array([20, 20])
def obj(xyz):
    x = xyz[:2]
    y = xyz[2:4]
    z = xyz[4:].reshape(6, 2)
    obj = 0
    for i in range(6):
        for j in range(2):
            obj = obj + z[i, j] * np.sqrt((x[j] - a[i]) ** 2 + (y[j] - b[i]) ** 2)
    return obj
con = []
con.append({"type": "eq", "fun": lambda z: z[4:].reshape(6, 2).sum(axis=1) - c})
con.append({"type": "ineq", "fun": lambda z: e - z[4:].reshape(6, 2).sum(axis=0)})
bd = [(0, np.inf) for i in range(16)]
res = minimize(obj, np.random.rand(16), constraints=con, bounds=bd)  # noqa: NPY002
print(res)
s = np.round(res.x, 4)
print("目标函数的最优值：", round(res.fun, 4))
print("x 的坐标为：", s[:2])
print("y 的坐标为：", s[2:4])
print("料场到工地的运输量为：\n", s[4:].reshape(6, 2).T)

## 3.9  双目标规划的三种解法

利润目标 $c_1^T x$（取最小即最大利润）与排污目标 $c_2^T x$（最小）的双目标线性规划，约束 $Ax \le b, x \ge 0$。解法一等权平均：最小化 $0.5(c_1+c_2)^T x$；解法二距离法：先分别求两目标各自最优值 $v_1, v_2$，再最小化到理想点的距离平方 $(c_1x-v_1)^2+(c_2x-v_2)^2$；解法三序贯（分层）法：在附加约束 $c_1^T x = v_1$ 下最小化 $c_2^T x$。分别用 `GLPK_MI` 与 `CVXOPT` 求解（源程序 ex5_9.py）。

In [ ]:
c1 = np.array([-2, -3])
c2 = np.array([1, 2])
a = np.array([[0.5, 0.25], [0.2, 0.2], [1, 5], [-1, -1]])
b = np.array([8, 4, 72, -10])
x = cp.Variable(2, pos=True)
obj = cp.Minimize(0.5 * (c1 + c2) @ x)
con = [a @ x <= b]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优解为：", x.value)
print("最优值为：", prob.value)
obj1 = cp.Minimize(c1 @ x)
prob1 = cp.Problem(obj1, con)
prob1.solve(solver="GLPK_MI")
v1 = prob1.value
obj2 = cp.Minimize(c2 @ x)
prob2 = cp.Problem(obj2, con)
prob2.solve(solver="GLPK_MI")
v2 = prob2.value
print("两个目标函数的最优值分别为：", v1, v2)
obj3 = cp.Minimize((c1 @ x - v1) ** 2 + (c2 @ x - v2) ** 2)
prob3 = cp.Problem(obj3, con)
prob3.solve(solver="CVXOPT")
print("解法二的最优解：", x.value)
con.append(c1 @ x == v1)
prob4 = cp.Problem(obj2, con)
prob4.solve(solver="GLPK_MI")
x3 = x.value
print("解法三的最优解：", x3)
print("利润：", -c1 @ x3)
print("排放污染物：", c2 @ x3)

## 3.10  带角度约束的偏差修正

给定 6 个点坐标 `x0, y0` 与方位角 `q`，先算两两欧氏距离阵 `d`（对角置 $\infty$）、方位角阵 `a0`、夹角阵 `b0`，分别写入 `data/5_10.xlsx` 的 sheet1、sheet2。再以偏差向量绝对值之和 $\sum|x_i|$ 为目标，用 `minimize` 在角度不等式约束、$x_i \in [-30,30]$ 边界下求最小修正量（随机初值出发），打印最优值与最优解（源程序 ex5_10.py）。

In [ ]:
x0 = np.array([150, 85, 150, 145, 130, 0])
y0 = np.array([140, 85, 155, 50, 150, 0])
q = np.array([243, 236, 220.5, 159, 230, 52])
d = np.zeros((6, 6))
a0 = np.zeros((6, 6))
b0 = np.zeros((6, 6))
xy0 = np.vstack([x0, y0]).T
for i in range(6):
    for j in range(6):
        d[i, j] = np.linalg.norm(xy0[i] - xy0[j])
d[d == 0] = np.inf
a0 = np.arcsin(8.0 / d) * 180 / np.pi
xy1 = x0 + 1j * y0
xy2 = np.exp(1j * q * np.pi / 180)
for m in range(6):
    for n in range(6):
        if n != m:
            b0[m, n] = np.angle((xy2[n] - xy2[m]) / (xy1[m] - xy1[n]))
b0 = b0 * 180 / np.pi
f = pd.ExcelWriter("data/5_10.xlsx")
pd.DataFrame(a0).to_excel(f, "sheet1", index=None)
pd.DataFrame(b0).to_excel(f, "sheet2", index=None)
f.save()


def obj(x):
    return sum(np.abs(x))


bd = [(-30, 30) for i in range(6)]
x0 = 30 * np.random.rand(6)  # noqa: NPY002
cons = [
    {
        "type": "ineq",
        "fun": lambda x: np.abs(b0[i, j] + (x[i] + x[j]) / 2) - a0[i, j],  # noqa: B023
    }
    for i in range(5)
    for j in range(i + 1, 6)
]
res = minimize(obj, x0, constraints=cons, bounds=bd)
print(res)
print("---------------")
print("目标函数的最优值：", round(res.fun, 4))
print("最优解为：", np.round(res.x, 4))